# Syndromes Discovery Notebook

**Source:** `bronze/source=qec_syndromes/syndromes_dataset.zip`  
**Goal:** Understand the raw data before building Silver.

Sections:
1. Bronze object and archive-member inventory
2. Schema and type profile
3. Missingness, duplicates, ranges, structural invariants
4. Candidate entities and keys
5. Evidence for rejected cross-source joins

In [1]:
import zipfile
import io
import csv
import ast

ZIP_PATH = '/course-data/raw/source=qec_syndromes/syndromes_dataset.zip'

## 1. Bronze object and archive-member inventory

In [2]:
# List all archive members with sizes
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        print(f"{info.filename}  --  {info.file_size:,} bytes")

d-3_pfr-0.000010_nb-10M.csv  --  4,411 bytes
d-3_pfr-0.000050_nb-10M.csv  --  13,715 bytes
d-3_pfr-0.000100_nb-10M.csv  --  31,115 bytes
d-3_pfr-0.000500_nb-10M.csv  --  89,438 bytes
d-3_pfr-0.001000_nb-10M.csv  --  181,222 bytes
d-3_pfr-0.005000_nb-10M.csv  --  1,323,765 bytes
d-3_pfr-0.010000_nb-10M.csv  --  3,148,485 bytes
README.txt  --  343 bytes


In [3]:
# Read the README inside the zip
with zipfile.ZipFile(ZIP_PATH) as zf:
    print(zf.read('README.txt').decode())

The file names: d-<surface_code_distance>_pfr-<physical_fault_rate>_nb-<number_of_samples>

The file format is a csv file with the following columns:
- label: binary label (0: no error, 1: error)
- syndromes: syndrome measurement sequence (tuples of the form (round, syndromes))
- quantity: number of samples for this label + syndrome sequence


**Finding:** README documents the column as `label` (singular), but the actual header is `labels` (plural). Verified below.

## 2. Schema and type profile

In [4]:
# Check actual header and first row using proper CSV parsing
# The syndromes field contains internal commas so naive split(',') would break it
with zipfile.ZipFile(ZIP_PATH) as zf:
    with zf.open('d-3_pfr-0.000010_nb-10M.csv') as f:
        reader = csv.reader(io.TextIOWrapper(f))
        header = next(reader)
        first_row = next(reader)

print("Header:", header)
print("First row:", first_row)

Header: ['labels', 'syndromes', 'quantity']
First row: ['0', '((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0))', '9987291']


In [5]:
# Parse the syndromes field and verify shape is 4 rounds x 4 checks
syndrome_raw = first_row[1]
parsed = ast.literal_eval(syndrome_raw)

print("Parsed type:", type(parsed))
print("Number of rounds:", len(parsed))
print("Checks per round:", len(parsed[0]))
print("Values:", parsed)

Parsed type: <class 'tuple'>
Number of rounds: 4
Checks per round: 4
Values: ((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0))


## 3. Missingness, duplicates, ranges, structural invariants

In [6]:
# Verify every syndrome in every file has shape (4, 4) with binary values only
issues = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for filename in [m for m in zf.namelist() if m.endswith('.csv')]:
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)
            for i, row in enumerate(reader):
                parsed = ast.literal_eval(row[1])
                if len(parsed) != 4 or any(len(r) != 4 for r in parsed):
                    issues.append(f"{filename} row {i}: wrong shape")
                if any(v not in (0, 1) for r in parsed for v in r):
                    issues.append(f"{filename} row {i}: non-binary value")

if issues:
    for issue in issues:
        print("ISSUE:", issue)
else:
    print("All syndromes are 4x4 with binary values.")

All syndromes are 4x4 with binary values.


In [7]:
# Verify quantity sums match the filename's nominal sample count (10M per file)
with zipfile.ZipFile(ZIP_PATH) as zf:
    for filename in sorted(m for m in zf.namelist() if m.endswith('.csv')):
        total = 0
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)
            for row in reader:
                total += int(row[2])
        print(f"{filename}: {total:,}")

d-3_pfr-0.000010_nb-10M.csv: 10,000,000
d-3_pfr-0.000050_nb-10M.csv: 10,000,000
d-3_pfr-0.000100_nb-10M.csv: 10,000,000
d-3_pfr-0.000500_nb-10M.csv: 10,000,000
d-3_pfr-0.001000_nb-10M.csv: 10,000,000
d-3_pfr-0.005000_nb-10M.csv: 10,000,000
d-3_pfr-0.010000_nb-10M.csv: 10,000,000


In [8]:
# Check for missing or empty values across all files
missing = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for filename in sorted(m for m in zf.namelist() if m.endswith('.csv')):
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)
            for i, row in enumerate(reader):
                if any(v.strip() == '' for v in row):
                    missing.append(f"{filename} row {i}")

if missing:
    for m in missing:
        print("MISSING:", m)
else:
    print("No missing values found.")

No missing values found.


In [9]:
# Check if the same syndrome appears under both labels (valid per spec, but verify it occurs)
with zipfile.ZipFile(ZIP_PATH) as zf:
    with zf.open('d-3_pfr-0.001000_nb-10M.csv') as f:
        reader = csv.reader(io.TextIOWrapper(f))
        next(reader)
        label0, label1 = set(), set()
        for row in reader:
            syndrome = row[1]
            if row[0] == '0':
                label0.add(syndrome)
            else:
                label1.add(syndrome)

overlap = label0 & label1
print(f"Syndromes appearing under both labels: {len(overlap)}")
if overlap:
    print("Example:", next(iter(overlap)))

Syndromes appearing under both labels: 467
Example: ((0, 1, 1, 0), (0, 0, 1, 0), (0, 1, 0, 0), (0, 1, 0, 0))


In [10]:
# Verify the natural key (syndrome, label) is unique within each file.
# Section 4 claims (experiment_id, syndrome_bits, logical_error_label) is the key,
# so no (syndrome, label) pair may repeat inside a single experiment file.
with zipfile.ZipFile(ZIP_PATH) as zf:
    for filename in sorted(m for m in zf.namelist() if m.endswith('.csv')):
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)
            seen = set()
            duplicates = 0
            for row in reader:
                key = (row[0], row[1])
                if key in seen:
                    duplicates += 1
                seen.add(key)
        status = "OK" if duplicates == 0 else f"{duplicates} DUPLICATE(S)"
        print(f"{filename}: {status}")

d-3_pfr-0.000010_nb-10M.csv: OK
d-3_pfr-0.000050_nb-10M.csv: OK
d-3_pfr-0.000100_nb-10M.csv: OK
d-3_pfr-0.000500_nb-10M.csv: OK
d-3_pfr-0.001000_nb-10M.csv: OK
d-3_pfr-0.005000_nb-10M.csv: OK
d-3_pfr-0.010000_nb-10M.csv: OK


In [11]:
# Reconcile the aggregate row count and confirm positive weights and fault-rate parsing.
# Cell [7] showed the *weighted* total (quantity sum) is 70M. This is the *row* count:
# the number of Silver rows this source produces. Also parse pfr and confirm quantity > 0.
import re

total_rows = 0
with zipfile.ZipFile(ZIP_PATH) as zf:
    for filename in sorted(m for m in zf.namelist() if m.endswith('.csv')):
        pfr = float(re.search(r'pfr-([0-9.]+)', filename).group(1))
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)
            rows = 0
            min_quantity = None
            for row in reader:
                rows += 1
                q = int(row[2])
                min_quantity = q if min_quantity is None else min(min_quantity, q)
        total_rows += rows
        print(f"{filename}: pfr={pfr:.6f}, rows={rows:,}, min_quantity={min_quantity}")

print(f"\nTotal aggregate rows (Silver row count): {total_rows:,}")

d-3_pfr-0.000010_nb-10M.csv: pfr=0.000010, rows=68, min_quantity=1
d-3_pfr-0.000050_nb-10M.csv: pfr=0.000050, rows=215, min_quantity=1
d-3_pfr-0.000100_nb-10M.csv: pfr=0.000100, rows=491, min_quantity=1
d-3_pfr-0.000500_nb-10M.csv: pfr=0.000500, rows=1,407, min_quantity=1
d-3_pfr-0.001000_nb-10M.csv: pfr=0.001000, rows=2,854, min_quantity=1
d-3_pfr-0.005000_nb-10M.csv: pfr=0.005000, rows=20,887, min_quantity=1
d-3_pfr-0.010000_nb-10M.csv: pfr=0.010000, rows=49,676, min_quantity=1

Total aggregate rows (Silver row count): 75,598


**Findings:** All 7 files pass shape validation (4 rounds × 4 binary values per round). Quantity sums equal exactly 10,000,000 per file (70M weighted observations total), and the aggregate **row** count is 75,598 across all files - this is the Silver row count for this source. The natural key `(syndrome, label)` is unique within every file (0 duplicates), confirming the Section 4 key. All seven physical fault rates parse cleanly from the filenames and every `quantity` is positive (min = 1). No missing or empty fields detected. 467 syndrome patterns appear under both labels in at least one file.


## 4. Candidate entities and keys

Each row represents one aggregate observation: a unique combination of experiment, syndrome pattern, and label.

- `experiment_id` is derived from the filename (encodes distance and physical fault rate)
- Syndrome alone is not a unique key: 467 syndromes appear under both labels in a single file
- The natural key for Silver is: `(experiment_id, syndrome_bits, logical_error_label)`
- `quantity` is a sample weight, not a row multiplier - do not expand into repeated rows

## 5. Evidence for rejected cross-source joins

The syndromes source cannot be joined row-by-row to the Google QEC or QASMBench sources:

- Syndromes are simulated aggregate counts; Google data is per hardware shot - no shared shot ID exists
- QASMBench contains circuit definitions with no labeled shots at all
- The sources share vocabulary (syndrome, parity, logical error) but no common record identifier

All three sources are stored in the same Gold database but kept as separate entities. No cross-source join is attempted.